# Challenge 10: Satellite Telemetry Anomaly Detection with K-NN and SMOTE

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
   - [4.4 Preprocessing](#44-preprocessing)
5. [Part 1: Baseline K-NN Without Imbalance Treatment](#5-part-1-baseline-k-nn-without-imbalance-treatment)
   - [5.1 Why the Default Metrics Mislead](#51-why-the-default-metrics-mislead)
   - [5.2 Training and Evaluating the Baseline](#52-training-and-evaluating-the-baseline)
   - [5.3 The Imbalance Problem in Detail](#53-the-imbalance-problem-in-detail)
6. [Part 2: Imbalance Treatment with SMOTE](#6-part-2-imbalance-treatment-with-smote)
   - [6.1 What SMOTE Does](#61-what-smote-does)
   - [6.2 Applying SMOTE to the Training Set](#62-applying-smote-to-the-training-set)
   - [6.3 K-NN After SMOTE](#63-k-nn-after-smote)
7. [Part 3: Hyperparameter Tuning on the Resampled Data](#7-part-3-hyperparameter-tuning-on-the-resampled-data)
   - [7.1 Grid Search with Cross-Validation](#71-grid-search-with-cross-validation)
   - [7.2 Final Evaluation and Comparison](#72-final-evaluation-and-comparison)
   - [7.3 Reflection Questions](#73-reflection-questions)
8. [Solution](#8-solution)
9. [References](#9-references)

---
## 1. Introduction

Modern Earth-observation satellites such as the ESA Sentinel series,
NASA's Landsat, and commercial platforms from Planet and Maxar operate
largely without real-time human supervision. Their onboard systems
and ground segment software must continuously monitor thousands of
telemetry parameters and classify the satellite's health state without
waiting for a human engineer to inspect each data packet.

The **SatelliteOps** dataset simulates the telemetry stream from an
Earth-observation satellite, sampled once per minute over approximately
seven days. Each sample represents a snapshot of ten onboard sensor
readings. The satellite can be in one of four operational states:

- **Nominal** (~59%): all systems within design tolerance. Normal
  operations, science data collection proceeding.
- **Thermal Anomaly** (~18%): one or more onboard components have
  exceeded temperature thresholds. Heaters have activated, increasing
  power consumption. The anomaly may be transient (eclipse transition)
  or sustained (thermal control failure).
- **Power Anomaly** (~14%): the solar array output is below expected
  levels, indicating panel degradation, a shadowing fault, or a power
  bus fault. Load shedding has been triggered to protect the battery.
- **Attitude Anomaly** (~8%): the attitude control system has detected
  pointing errors above threshold. The reaction wheels are working to
  correct the error but are approaching saturation. This is the
  smallest class and the most operationally dangerous: uncontrolled
  attitude drift can prevent science data downlink, cause antenna
  mispointing, and in severe cases trigger a safe mode entry.

The class imbalance is intentional and realistic. Satellite anomalies
are rare by design: the systems are engineered with large safety margins.
But the rarity of anomalies creates a machine learning problem that this
challenge addresses directly. A K-NN classifier trained on the imbalanced
dataset will have a strong bias toward the Nominal class and will miss
a disproportionate fraction of the attitude anomalies, which are the
most critical events. You will quantify this failure, apply SMOTE to
correct it, and use appropriate evaluation metrics to assess the result.

### Learning Objectives

- Apply `KNeighborsClassifier` to a multi-class imbalanced telemetry
  classification problem
- Recognise why accuracy is a misleading metric for imbalanced datasets
  and use macro-averaged F1 and Matthews Correlation Coefficient instead
- Apply SMOTE (Synthetic Minority Over-sampling Technique) correctly
  to the training set only, and understand what it does to the data
- Use `GridSearchCV` with macro-F1 scoring to tune k and other KNN
  hyperparameters on the SMOTE-resampled training set
- Compare baseline and SMOTE-tuned classifiers using per-class precision,
  recall, and the confusion matrix, and articulate the precision-recall
  tradeoff that SMOTE introduces

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `KNeighborsClassifier` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html |
| `SMOTE` (imbalanced-learn) | https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html |
| imbalanced-learn user guide | https://imbalanced-learn.org/stable/user_guide.html |
| `GridSearchCV` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html |
| `matthews_corrcoef` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.matthews_corrcoef.html |
| `classification_report` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html |
| ESA Sentinel satellite operations | https://www.esa.int/Applications/Observing_the_Earth/Copernicus/Sentinel-2 |
| Chawla et al. (2002) SMOTE paper | https://doi.org/10.1613/jair.953 |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook requires Python 3.9 or later, scikit-learn 1.0 or later,
and `imbalanced-learn` (the `imblearn` package). Install imbalanced-learn
if it is not already available:

```
pip install imbalanced-learn
```

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations |
| `pandas` | Loading and inspecting data |
| `matplotlib` | Plotting |
| `seaborn` | Heatmaps and distribution plots |
| `sklearn.preprocessing` | `StandardScaler` |
| `sklearn.neighbors` | `KNeighborsClassifier` |
| `sklearn.model_selection` | `train_test_split`, `GridSearchCV`, `StratifiedKFold` |
| `sklearn.metrics` | `classification_report`, `matthews_corrcoef`, `f1_score`, `ConfusionMatrixDisplay` |
| `imblearn.over_sampling` | `SMOTE` |

### 3.3 Data

| Property | Value |
|---|---|
| Filename | `satellite.csv` |
| Location | `data/satellite.csv` (relative to this notebook) |
| Total rows | 10,000 |
| Features | 10 telemetry measurements |
| Target column | `operational_state` |
| Class 0 | nominal: ~5,884 samples (~58.8%) |
| Class 1 | thermal\_anomaly: ~1,812 samples (~18.1%) |
| Class 2 | power\_anomaly: ~1,430 samples (~14.3%) |
| Class 3 | attitude\_anomaly: ~874 samples (~8.7%) |

The attitude anomaly class is the most operationally critical and the
least represented. A classifier that always predicts Nominal achieves
~58.8% accuracy while detecting zero anomalies. This is the imbalance
problem in its clearest form.

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from imblearn.over_sampling import SMOTE

CLASS_NAMES  = ['nominal', 'thermal_anomaly', 'power_anomaly', 'attitude_anomaly']
FEATURE_COLS = [
    'battery_voltage_v', 'solar_panel_current_a', 'cpu_temp_c',
    'reaction_wheel_rpm', 'attitude_error_roll_deg',
    'attitude_error_pitch_deg', 'attitude_error_yaw_deg',
    'link_quality_db', 'power_consumption_w', 'eclipse_flag'
]

print('Libraries loaded successfully.')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/satellite.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(8)

### 4.2 Understanding the Features

| Feature | Units | Description |
|---|---|---|
| `battery_voltage_v` | Volts | Battery terminal voltage. Drops during eclipse (~28-29V) and recovers in sunlight (~30-32V). Low values (<26V) indicate Power Anomaly. Correlated with `solar_panel_current_a`. |
| `solar_panel_current_a` | Amperes | Solar array output current. Near zero during eclipse. Reduced below expected indicates panel degradation (Power Anomaly). Strongly correlated with `battery_voltage_v`. |
| `cpu_temp_c` | deg C | Onboard processor temperature. Nominal: 15-45 degC. Above 52 degC: Thermal Anomaly. Correlated with `power_consumption_w`. |
| `reaction_wheel_rpm` | rpm | Reaction wheel spin rate for attitude control. Nominal: -3000 to +3000 rpm. Elevated absolute values indicate Attitude Anomaly. |
| `attitude_error_roll_deg` | degrees | Roll axis pointing error. Nominal: <0.05 deg. Elevated values indicate Attitude Anomaly. Correlated with pitch and yaw errors. |
| `attitude_error_pitch_deg` | degrees | Pitch axis pointing error. Correlated with roll error. |
| `attitude_error_yaw_deg` | degrees | Yaw axis pointing error. Correlated with pitch error. |
| `link_quality_db` | dBm (relative) | Downlink signal quality. Degrades under antenna mispointing (Attitude Anomaly). Largely independent of power/thermal features. |
| `power_consumption_w` | Watts | Total satellite power consumption. Elevated during Thermal Anomaly (heaters), reduced during Power Anomaly (load shedding). Correlated with `cpu_temp_c`. |
| `eclipse_flag` | binary | 1 = satellite in Earth's shadow, 0 = sunlit. Strongly correlated with `solar_panel_current_a` and `battery_voltage_v`. |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
print('Class distribution:')
counts = df['operational_state'].value_counts().sort_index()
for k, v in counts.items():
    print(f'  Class {k} ({CLASS_NAMES[k]:20s}): {v:5d}  ({100*v/len(df):.1f}%)')

print()
print('A trivial classifier that always predicts Nominal would score:')
trivial_acc = counts[0] / len(df)
print(f'  Accuracy: {trivial_acc:.4f}  ({100*trivial_acc:.1f}%)')
print('  But it would detect ZERO anomalies.')

**This is the imbalance problem stated clearly.** The ~59% majority class
provides a deceptively high accuracy floor that disguises complete failure
to detect any anomaly. Accuracy is the wrong metric here.

In [ ]:
# Listing 4.
print('Mean feature values by operational state:')
print(df.groupby('operational_state')[FEATURE_COLS].mean().round(2).to_string())

**Questions to consider:**

- Which features clearly differ between Nominal and each anomaly class?
  `cpu_temp_c` and `power_consumption_w` should be elevated for Thermal
  Anomaly. `battery_voltage_v` and `solar_panel_current_a` should be
  depressed for Power Anomaly. What distinguishes Attitude Anomaly?
- The `attitude_error_*` features have very similar means across all
  classes. Does this mean they are not useful for classification?
  (Hint: check the standard deviations too.)
- `eclipse_flag` has a mean around 0.35 for all classes. Does this
  make sense? Why should the eclipse fraction be similar regardless
  of operational state?

In [ ]:
# Listing 5.
# Distribution plots for the most discriminating features
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()
colours = {0:'steelblue', 1:'firebrick', 2:'seagreen', 3:'darkorange'}

for ax, feat in zip(axes, FEATURE_COLS):
    for cls in range(4):
        subset = df.loc[df['operational_state'] == cls, feat]
        subset.plot(kind='kde', ax=ax, label=CLASS_NAMES[cls],
                    color=colours[cls], linewidth=1.8)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
    ax.legend(fontsize=7)

plt.suptitle('Feature Distributions by Operational State (SatelliteOps)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Questions to consider:**

- The `attitude_error_roll_deg` distribution for Attitude Anomaly has
  the same centre as Nominal but a much wider spread. The KDE plots
  may show significant overlap near zero. What does this tell you about
  how difficult the Attitude Anomaly class is to detect?
- `cpu_temp_c` shows a clear separation between Thermal Anomaly
  (high temperature) and Power Anomaly (low temperature from reduced
  processing load). Will KNN have difficulty separating these two classes?
- `eclipse_flag` is binary (0 or 1). Its KDE will show two spikes.
  Does the eclipse flag provide any discrimination between the four classes?

In [ ]:
# Listing 6.
# Correlation matrix
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix (SatelliteOps)', fontsize=12)
plt.tight_layout()
plt.show()

**Questions to consider:**

- `battery_voltage_v` and `solar_panel_current_a` are strongly correlated
  (approx. 0.66). Both drop during eclipse. `eclipse_flag` is strongly negatively
  correlated with both (approx. -0.60, -0.88). These three features form a
  physically correlated power subsystem group.
- `cpu_temp_c` and `power_consumption_w` are strongly correlated (~0.83).
  This makes physical sense: more processing generates more heat.
- The attitude error features (`roll`, `pitch`, `yaw`) are moderately
  correlated with each other (approx. 0.26-0.54), reflecting that a disturbance
  torque typically affects multiple axes.
- `link_quality_db` and `reaction_wheel_rpm` are largely uncorrelated
  with the power and thermal features, providing independent information.

KNN uses Euclidean distance. Correlated features can distort the distance
metric: if `battery_voltage_v` and `solar_panel_current_a` are highly
correlated, the power subsystem effectively gets double-counted in the
distance. `StandardScaler` addresses scale but not correlation; PCA
would address both. For this challenge, StandardScaler is sufficient.

### 4.4 Preprocessing

In [ ]:
# Listing 7.
# TODO: Preprocessing.
#
# 1. Separate features (X) from the target ('operational_state') (y).
#
# 2. Split into training and test sets.
#    Use an 80/20 split, random_state=42, stratify=y.
#    Print class counts in y_train and y_test.
#    Check that the minority class (attitude_anomaly) is represented
#    proportionally in both splits.
#
# 3. Apply StandardScaler.
#    Fit on X_train, transform both.
#    Store as X_train_sc and X_test_sc.
#
# Note: SMOTE will be applied to the SCALED training set later.
# Do not apply SMOTE before splitting or before scaling.
# Applying SMOTE to the test set would be a serious data leakage error.

# YOUR CODE HERE

---
## 5. Part 1: Baseline K-NN Without Imbalance Treatment

### 5.1 Why the Default Metrics Mislead

Before training any model, compute the "naive" accuracy: the accuracy
of a classifier that always predicts the majority class (Nominal).
This is your minimum benchmark. Any model that does not substantially
exceed this on minority-class recall is not learning anything useful.

The appropriate metrics for an imbalanced multi-class problem are:

- **Per-class recall**: for each class, what fraction of true positives
  did the model find? Attitude anomaly recall is the most critical:
  a missed attitude anomaly may not be caught until the satellite
  drifts into an uncontrolled state.
- **Macro-averaged F1**: the unweighted mean of per-class F1 scores.
  This gives equal weight to each class regardless of size, penalising
  models that ignore minority classes.
- **Matthews Correlation Coefficient (MCC)**: a single number summarising
  the quality of the classification across all classes. MCC of 0 means
  the model is no better than random; MCC of 1 is perfect. MCC handles
  imbalance naturally and is less sensitive to majority class dominance
  than accuracy.

### 5.2 Training and Evaluating the Baseline

In [ ]:
# Listing 8.
# TODO: Baseline KNN.
#
# 1. Instantiate KNeighborsClassifier(n_neighbors=5).
#    Fit on X_train_sc. Predict on X_test_sc.
#
# 2. Compute and print:
#    - Overall accuracy
#    - Macro-averaged F1 score (f1_score with average='macro')
#    - Matthews Correlation Coefficient (matthews_corrcoef)
#    - Full classification_report with target_names=CLASS_NAMES
#
# 3. Plot the confusion matrix using ConfusionMatrixDisplay.from_predictions().
#    Use display_labels=CLASS_NAMES and rotate x-axis labels 45 degrees.
#
# 4. Identify the row for attitude_anomaly in the confusion matrix.
#    How many true attitude anomalies are being predicted as Nominal?
#    What is the recall for attitude_anomaly?

# YOUR CODE HERE

### 5.3 The Imbalance Problem in Detail

In [ ]:
# Listing 9.
# TODO: Visualise the class imbalance problem.
#
# Create a bar chart showing per-class recall from the baseline KNN.
# Extract per-class recall from:
#   report = classification_report(y_test, y_pred, output_dict=True)
#   recall_per_class = [report[str(i)]['recall'] for i in range(4)]
#
# Plot with CLASS_NAMES on x-axis, recall on y-axis.
# Add a horizontal line at the macro recall (mean of per-class recalls).
# Colour the attitude_anomaly bar red to highlight the problem.
# Title: 'Per-class Recall: Baseline KNN (k=5, no imbalance treatment)'
#
# The visual should make clear that attitude_anomaly recall is
# substantially below the other classes.

# YOUR CODE HERE

---
## 6. Part 2: Imbalance Treatment with SMOTE

### 6.1 What SMOTE Does

SMOTE (Synthetic Minority Over-sampling Technique, Chawla et al. 2002)
addresses class imbalance by generating synthetic training examples for
the minority classes rather than simply duplicating existing ones. For
each minority class sample, SMOTE selects k nearest neighbours in the
same class and creates new samples along the line segments connecting
the sample to its neighbours. This populates the minority class regions
of feature space with plausible new training points.

Three rules for applying SMOTE correctly:

1. **Apply to training data only.** Never apply SMOTE to the test set.
   The test set must represent the real, imbalanced distribution that
   the deployed model will encounter. If you SMOTE the test set, your
   evaluation is not realistic.
2. **Apply after scaling.** SMOTE interpolates between points in feature
   space. If features are on different scales, the interpolation is
   distorted. Scale first, then SMOTE.
3. **Apply after splitting.** Applying SMOTE before splitting allows
   synthetic test points to be generated from training-set neighbours,
   creating data leakage.

After SMOTE, all four classes will have equal representation in the
training set. The KNN classifier trained on this balanced set will give
equal weight to each class when making predictions, improving minority
class recall at the cost of some majority class precision.

### 6.2 Applying SMOTE to the Training Set

In [ ]:
# Listing 10.
# TODO: Apply SMOTE to the scaled training set.
#
# 1. Instantiate SMOTE(random_state=42).
#    Apply: X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)
#
# 2. Print the class distribution in y_train_res.
#    All four classes should now have equal counts.
#    What is the total training set size after SMOTE?
#
# 3. Create a bar chart comparing class counts BEFORE and AFTER SMOTE.
#    Use grouped bars (two bars per class: original and resampled).
#    This shows what SMOTE has done to the training distribution.
#
# 4. In a comment or markdown cell, answer:
#    SMOTE adds synthetic Attitude Anomaly samples. These synthetic points
#    were generated by interpolating between real attitude anomaly samples.
#    In the physical context of satellite telemetry, what does this mean?
#    Are the synthetic points physically plausible?

# YOUR CODE HERE

### 6.3 K-NN After SMOTE

In [ ]:
# Listing 11.
# TODO: Train KNN on the SMOTE-resampled training set.
#
# 1. Instantiate KNeighborsClassifier(n_neighbors=5).
#    Fit on X_train_res and y_train_res (the SMOTE-resampled data).
#    Predict on X_test_sc (the ORIGINAL test set, not resampled).
#
# 2. Compute and print:
#    - Overall accuracy
#    - Macro-averaged F1 score
#    - MCC
#    - Full classification_report
#
# 3. Compare to the baseline (Section 5.2):
#    - Did attitude_anomaly recall improve?
#    - Did nominal precision change? In which direction?
#    - Did overall accuracy increase or decrease?
#    - Did macro-F1 increase or decrease?
#
# 4. Plot the confusion matrix and compare to the baseline.

# YOUR CODE HERE

---
## 7. Part 3: Hyperparameter Tuning on the Resampled Data

K-NN has several hyperparameters that can meaningfully affect performance
on imbalanced data:

- **`n_neighbors` (k)**: the number of neighbours used for voting.
  Larger k smooths the decision boundary but may absorb minority class
  regions back into the majority. Smaller k captures local structure
  but is noisier.
- **`weights`**: `'uniform'` gives equal weight to all k neighbours;
  `'distance'` weights closer neighbours more heavily. Distance weighting
  can help near class boundaries where the minority class has been
  over-sampled.
- **`metric`**: the distance metric. `'euclidean'` is standard; `'manhattan'`
  (L1 distance) can be more robust to outliers and performs well on
  high-dimensional or mixed-scale data.

Use `GridSearchCV` with `scoring='f1_macro'` (not accuracy) to select
the best configuration. This ensures the search optimises for balanced
performance across all four classes, not just majority-class accuracy.

### 7.1 Grid Search with Cross-Validation

In [ ]:
# Listing 12.
# TODO: Grid search on SMOTE-resampled training data.
#
# 1. Define the parameter grid:
#    param_grid = {
#        'n_neighbors': [3, 5, 7, 11, 15],
#        'weights':     ['uniform', 'distance'],
#        'metric':      ['euclidean', 'manhattan'],
#    }
#
# 2. Create a StratifiedKFold(n_splits=5, shuffle=True, random_state=42).
#
# 3. Create GridSearchCV:
#    gs = GridSearchCV(
#        estimator=KNeighborsClassifier(),
#        param_grid=param_grid,
#        cv=cv,
#        scoring='f1_macro',
#        n_jobs=-1,
#        verbose=1,
#        refit=True
#    )
#    Fit on X_train_res and y_train_res.
#
# 4. Print gs.best_params_ and gs.best_score_.
#
# 5. Print the top 5 configurations from gs.cv_results_ sorted by
#    mean_test_score descending.

# YOUR CODE HERE

### 7.2 Final Evaluation and Comparison

In [ ]:
# Listing 13.
# TODO: Evaluate the best model and compare all three configurations.
#
# 1. Use gs.best_estimator_ to predict on X_test_sc.
#
# 2. Compute accuracy, macro-F1, and MCC for the tuned model.
#
# 3. Create a comparison table (as a DataFrame) with three rows:
#    - Baseline KNN (k=5, no SMOTE)
#    - SMOTE KNN (k=5, no tuning)
#    - Tuned SMOTE KNN (best params from grid search)
#    Columns: accuracy, macro_f1, mcc, attitude_recall, attitude_precision
#    Print the table.
#
# 4. Create a side-by-side confusion matrix plot (1 row, 3 columns)
#    showing the confusion matrices for all three models.
#    Use the same colour scale for all three for fair comparison.
#
# 5. Answer in a markdown cell:
#    - Which model gives the best attitude_anomaly recall?
#    - Which model gives the best macro-F1?
#    - Is there a tradeoff? If SMOTE improves attitude recall, what
#      does it cost in nominal precision?
#    - In the satellite operations context, which error type is more
#      acceptable: a missed attitude anomaly, or a false alarm?

# YOUR CODE HERE

### 7.3 Reflection Questions

1. SMOTE generates synthetic samples by interpolating between real
   minority class neighbours. In the satellite telemetry context,
   a synthetic Attitude Anomaly sample is a made-up set of sensor
   readings that never actually occurred. Is this a valid approach?
   What assumption about the data distribution does SMOTE make,
   and when might that assumption be violated?

2. After SMOTE, the training set has equal class counts. But the real
   world has 8.7% attitude anomalies, not 25%. The model trained on
   the balanced set will implicitly assign a uniform prior to all
   classes. How could you inform the classifier of the true class
   frequencies without changing the training data?
   (Hint: consider the `predict_proba` output and a threshold adjustment,
   or look at the `class_weight` parameter in other classifiers.)

3. The grid search used `scoring='f1_macro'`. If you changed the scoring
   to `'recall_macro'`, would you expect the best `n_neighbors` to
   increase or decrease? Why? (Hint: think about what larger k does
   to the decision boundary near the minority class region.)

4. The Matthew Correlation Coefficient (MCC) is often described as
   a more informative single-number metric than accuracy or macro-F1
   for imbalanced problems. Looking at your three models, does MCC
   tell a different story than macro-F1? Which do you trust more for
   this specific problem, and why?

5. SMOTE is one approach to imbalance. Two alternatives are:
   (a) Cost-sensitive learning: assign higher misclassification costs
       to the minority class (available in many classifiers via
       `class_weight='balanced'`).
   (b) Ensemble methods: BalancedRandomForest or EasyEnsemble from
       imbalanced-learn.
   Without implementing them, which of these alternatives might work
   better than SMOTE + KNN for this specific problem? Explain your
   reasoning in terms of the class structure you observed.

---
## 8. Solution

**Read this section only after attempting the challenge yourself.**

### Step 1: Preprocessing

In [ ]:
# Listing 14.
X = df[FEATURE_COLS].values
y = df['operational_state'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Class counts in training set:')
for cls in range(4):
    n = int(np.sum(y_train == cls))
    print(f'  {CLASS_NAMES[cls]:22s}: {n:5d}  ({100*n/len(y_train):.1f}%)')

print(f'\nTraining set:  {X_train_sc.shape}')
print(f'Test set:      {X_test_sc.shape}')

### Step 2: Baseline KNN

In [ ]:
# Listing 15.
knn_base = KNeighborsClassifier(n_neighbors=5)
knn_base.fit(X_train_sc, y_train)
y_pred_base = knn_base.predict(X_test_sc)

acc_base = accuracy_score(y_test, y_pred_base)
f1_base  = f1_score(y_test, y_pred_base, average='macro')
mcc_base = matthews_corrcoef(y_test, y_pred_base)
report_base = classification_report(y_test, y_pred_base, output_dict=True)
att_recall_base = report_base['3']['recall']
att_prec_base   = report_base['3']['precision']

print('Baseline KNN (k=5, no imbalance treatment):')
print(f'  Accuracy:          {acc_base:.4f}')
print(f'  Macro-F1:          {f1_base:.4f}')
print(f'  MCC:               {mcc_base:.4f}')
print(f'  Attitude recall:   {att_recall_base:.4f}')
print()
print(classification_report(y_test, y_pred_base, target_names=CLASS_NAMES))

The baseline KNN achieves 91% overall accuracy but misses roughly 35%
of attitude anomalies (recall ~0.65). The model has a strong bias
toward Nominal: when a sample is ambiguous, the majority class wins
the k=5 neighbourhood vote because there are many more Nominal training
examples in every region of feature space. An attitude anomaly surrounded
by 5 neighbours that are mostly Nominal (because of class imbalance)
will be misclassified as Nominal even if its attitude error features are
elevated.

In [ ]:
# Listing 16.
# Per-class recall bar chart
recall_per_class = [report_base[str(i)]['recall'] for i in range(4)]
colours_bar = ['firebrick' if i == 3 else 'steelblue' for i in range(4)]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, recall_per_class, color=colours_bar)
ax.axhline(np.mean(recall_per_class), color='black', linestyle='--',
           linewidth=1.5, label=f'Macro recall ({np.mean(recall_per_class):.3f})')
ax.set_ylabel('Recall', fontsize=11)
ax.set_title('Per-class Recall: Baseline KNN (k=5, no SMOTE)', fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

### Step 3: SMOTE

In [ ]:
# Listing 17.
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)

print('Class counts BEFORE SMOTE (training set):')
for cls in range(4):
    n = int(np.sum(y_train == cls))
    print(f'  {CLASS_NAMES[cls]:22s}: {n:5d}')

print(f'\nClass counts AFTER SMOTE (resampled training set):')
for cls in range(4):
    n = int(np.sum(y_train_res == cls))
    print(f'  {CLASS_NAMES[cls]:22s}: {n:5d}')

print(f'\nTotal training samples: {len(y_train)} -> {len(y_train_res)}')
n_synthetic = len(y_train_res) - len(y_train)
print(f'Synthetic samples generated: {n_synthetic}')

In [ ]:
# Listing 18.
# Visualise class distribution before and after SMOTE
counts_before = [int(np.sum(y_train == c)) for c in range(4)]
counts_after  = [int(np.sum(y_train_res == c)) for c in range(4)]
x = np.arange(4); width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, counts_before, width, label='Before SMOTE', color='steelblue')
ax.bar(x + width/2, counts_after,  width, label='After SMOTE',  color='firebrick', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=20, ha='right')
ax.set_ylabel('Number of training samples', fontsize=11)
ax.set_title('Training Set Class Distribution Before and After SMOTE', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Step 4: KNN After SMOTE

In [ ]:
# Listing 19.
knn_smote = KNeighborsClassifier(n_neighbors=5)
knn_smote.fit(X_train_res, y_train_res)
y_pred_smote = knn_smote.predict(X_test_sc)

acc_smote = accuracy_score(y_test, y_pred_smote)
f1_smote  = f1_score(y_test, y_pred_smote, average='macro')
mcc_smote = matthews_corrcoef(y_test, y_pred_smote)
report_smote = classification_report(y_test, y_pred_smote, output_dict=True)
att_recall_smote = report_smote['3']['recall']
att_prec_smote   = report_smote['3']['precision']

print('SMOTE KNN (k=5, SMOTE applied):')
print(f'  Accuracy:          {acc_smote:.4f}  (was {acc_base:.4f})')
print(f'  Macro-F1:          {f1_smote:.4f}  (was {f1_base:.4f})')
print(f'  MCC:               {mcc_smote:.4f}  (was {mcc_base:.4f})')
print(f'  Attitude recall:   {att_recall_smote:.4f}  (was {att_recall_base:.4f})')
print()
print(classification_report(y_test, y_pred_smote, target_names=CLASS_NAMES))

**Reading the SMOTE result:**

Attitude anomaly recall jumps from ~65% to ~82%: the model now finds
more genuine attitude anomalies. However, overall accuracy drops from
~91% to ~86%, and nominal precision also drops. This is the
precision-recall tradeoff that SMOTE introduces: by filling the attitude
anomaly region of feature space with synthetic training points, the model
becomes more aggressive in labelling samples as attitude anomalies,
including some that are actually nominal (false alarms).

Whether this tradeoff is acceptable depends on operational context:
in satellite operations, a missed attitude anomaly (false negative) is
far more costly than a false alarm (false positive). A false alarm
triggers an operator review; a missed anomaly may result in uncontrolled
satellite drift. The SMOTE model is therefore operationally preferable
even though its overall accuracy is lower.

### Step 5: Grid Search

In [ ]:
# Listing 20.
param_grid = {
    'n_neighbors': [3, 5, 7, 11, 15],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan'],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=0,
    refit=True
)
gs.fit(X_train_res, y_train_res)

print(f'Best parameters: {gs.best_params_}')
print(f'Best CV macro-F1: {gs.best_score_:.4f}')

results_df = (pd.DataFrame(gs.cv_results_)
              [['param_n_neighbors', 'param_weights', 'param_metric',
                'mean_test_score', 'std_test_score']]
              .sort_values('mean_test_score', ascending=False)
              .head(8))
print('\nTop 8 configurations by CV macro-F1:')
print(results_df.to_string(index=False))

**Reading the grid search results:**

The best configuration typically uses larger k (11-15) and distance
weighting with Manhattan metric. Larger k reduces the noise from
individual synthetic SMOTE points: with k=5, the model can be swayed
by a small cluster of synthetic attitude anomaly points; with k=15,
it requires broader neighbourhood consensus before assigning a minority
class label. Distance weighting ensures that closer (more physically
similar) neighbours have more influence, which is particularly useful
near the attitude anomaly / nominal boundary where the two classes
genuinely overlap.

### Step 6: Final Comparison

In [ ]:
# Listing 21.
y_pred_tuned = gs.best_estimator_.predict(X_test_sc)
acc_tuned = accuracy_score(y_test, y_pred_tuned)
f1_tuned  = f1_score(y_test, y_pred_tuned, average='macro')
mcc_tuned = matthews_corrcoef(y_test, y_pred_tuned)
report_tuned = classification_report(y_test, y_pred_tuned, output_dict=True)
att_recall_tuned = report_tuned['3']['recall']
att_prec_tuned   = report_tuned['3']['precision']

comparison = pd.DataFrame({
    'Model':              ['Baseline KNN (k=5)',
                           'SMOTE KNN (k=5)',
                           f'Tuned SMOTE KNN {gs.best_params_}'],
    'Accuracy':           [acc_base,  acc_smote,  acc_tuned],
    'Macro-F1':           [f1_base,   f1_smote,   f1_tuned],
    'MCC':                [mcc_base,  mcc_smote,  mcc_tuned],
    'Att Recall':         [att_recall_base, att_recall_smote, att_recall_tuned],
    'Att Precision':      [att_prec_base,   att_prec_smote,   att_prec_tuned],
})
print('Model comparison:')
print(comparison.round(4).to_string(index=False))

In [ ]:
# Listing 22.
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
preds = [y_pred_base, y_pred_smote, y_pred_tuned]
titles = ['Baseline KNN (k=5)', 'SMOTE KNN (k=5)',
          f'Tuned SMOTE KNN\n{gs.best_params_}']

for ax, yp, title in zip(axes, preds, titles):
    ConfusionMatrixDisplay.from_predictions(
        y_test, yp, display_labels=CLASS_NAMES,
        cmap='Blues', ax=ax, colorbar=False
    )
    ax.set_title(title, fontsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Confusion Matrix Comparison: SatelliteOps', fontsize=13)
plt.tight_layout()
plt.show()

**Summary of findings:**

The tuned SMOTE KNN achieves the best balance across all metrics. The
comparison table and confusion matrices tell a clear story:

- **Baseline KNN**: highest accuracy, highest nominal precision, lowest
  attitude recall (~65%). Operationally unacceptable: one in three
  attitude anomalies is missed.
- **SMOTE KNN (k=5)**: attitude recall jumps to ~82%. Nominal precision
  drops (more false alarms). Overall accuracy falls.
- **Tuned SMOTE KNN**: best macro-F1 and MCC. Attitude recall similar
  to SMOTE KNN but nominal precision partially recovers due to the
  smoothing effect of larger k and distance weighting.

The MCC values confirm this ordering: tuned SMOTE KNN has the highest
MCC, meaning it has the most balanced performance across all four classes.
Accuracy would have led us to keep the baseline, which is the worst
model for the actual operational requirement.

The key lesson: **the right metric drives the right choice.** Optimising
accuracy on an imbalanced dataset selects for majority class performance.
Optimising macro-F1 or MCC selects for balanced performance across all
classes including the rare but critical minority.

---
## 9. References

1. Chawla, N.V., Bowyer, K.W., Hall, L.O., and Kegelmeyer, W.P. (2002).
   SMOTE: Synthetic Minority Over-sampling Technique.
   *Journal of Artificial Intelligence Research*, 16, 321-357.
   https://doi.org/10.1613/jair.953
   Original paper introducing SMOTE. The algorithm used in this challenge
   is the version implemented in the imbalanced-learn library.

2. Lemaitre, G., Nogueira, F., and Aridas, C.K. (2017).
   Imbalanced-learn: A Python Toolbox to Tackle the Curse of Imbalanced
   Datasets in Machine Learning.
   *Journal of Machine Learning Research*, 18(17), 1-5.
   https://jmlr.org/papers/v18/16-365.html
   Reference for the imbalanced-learn library used for SMOTE in this challenge.

3. Matthews, B.W. (1975). Comparison of the predicted and observed
   secondary structure of T4 phage lysozyme.
   *Biochimica et Biophysica Acta*, 405(2), 442-451.
   Original paper introducing the Matthews Correlation Coefficient.

4. He, H. and Garcia, E.A. (2009). Learning from imbalanced data.
   *IEEE Transactions on Knowledge and Data Engineering*, 21(9), 1263-1284.
   Comprehensive survey of imbalanced learning techniques including
   oversampling, undersampling, cost-sensitive learning, and ensemble methods.

5. ESA Sentinel-2 Mission Guide. European Space Agency.
   https://sentinel.esa.int/web/sentinel/missions/sentinel-2
   Reference for the operational context of Earth-observation satellite
   telemetry monitoring that motivates this challenge.

6. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html